In [2]:
import pandas as pd
import json

In [142]:
model_id = "llama-3.3-70b"
limit = True
use_probabilistic = False

df_scores = pd.read_csv(f"output_scores/{model_id}{'_limit' if limit else ''}_parsed.csv")
model_alignment_data = json.load(open(f"model_alignment{'_probability' if use_probabilistic else ''}/{model_id}.json"))
df_scores = df_scores.drop(columns=["vrd_1", "text_1", "explanation_1", "generated_opinion_1", "model_used_1", "vrd_2", "text_2", "explanation_2", "generated_opinion_2", "model_used_2", "generated_score_1to2", "generated_score_2to1", "model_used"])

In [144]:
model_alignment_data_reformatted = {}
for situation, entry in model_alignment_data.items():
    p_acceptable = entry["p(ACCEPTABLE)"]
    p_unacceptable = entry["p(UNACCEPTABLE)"]
    model_alignment_data_reformatted[situation] = {
        "p_acceptable": p_acceptable,
        "p_unacceptable": p_unacceptable,
        "majority_opinion": "Supports" if p_acceptable >= p_unacceptable else "Opposes",
        "minority_opinion": "Supports" if p_acceptable < p_unacceptable else "Opposes",
        "majority_opinion_probability": max(p_acceptable, p_unacceptable),
        "minority_opinion_probability": min(p_acceptable, p_unacceptable),
    }

In [145]:
# Enrich scores df with model alignment dat
# For each situation, add majoirity opiniony and majority opinion probability
majority_opinions = []
minority_opinions = []
majority_opinions_probs = []
minority_opinions_probs = []
for idx, row in df_scores.iterrows():
    situation = row["situation"]
    majority_opinion = model_alignment_data_reformatted[situation]["majority_opinion"]
    majority_opinion_prob = model_alignment_data_reformatted[situation]["majority_opinion_probability"]
    minority_opinion = model_alignment_data_reformatted[situation]["minority_opinion"]
    minority_opinion_prob = model_alignment_data_reformatted[situation]["minority_opinion_probability"]
    majority_opinions.append(majority_opinion)
    majority_opinions_probs.append(majority_opinion_prob)
    minority_opinions.append(minority_opinion)
    minority_opinions_probs.append(minority_opinion_prob)

df_scores["majority_opinion"] = majority_opinions
df_scores["majority_opinion_probability"] = majority_opinions_probs
df_scores["minority_opinion"] = minority_opinions
df_scores["minority_opinion_probability"] = minority_opinions_probs

### Cases of interest Support vs Oppose

In [146]:
# Other options (both either, one either, both the same valence)
df_other = df_scores[(df_scores["valence_1"] == df_scores["valence_2"]) | ((df_scores["valence_1"] != df_scores["minority_opinion"]) & (df_scores["valence_2"] != df_scores["minority_opinion"]))]
# All other rows, row index use
df_interest = df_scores[df_scores.index.isin(df_other.index) == False]

In [147]:
# Count reverse scores in first and second df
def count_reverse_scores(df):
    cnt_reverse = 0
    for idx, row in df.iterrows():
        if row["parsed_score1to2"] != row["parsed_score2to1"]:
            cnt_reverse += 1
    return cnt_reverse, round(cnt_reverse * 100 / len(df), 2)

In [148]:
all_scores_safe_unsafe = []
all_scores_unsafe_safe = []
for idx, row in df_interest.iterrows():
    minority_opinion = row["minority_opinion"] # Proxy for danger
    idx_minority_opinion = "1" if row['valence_1'] == minority_opinion else "2"
    idx_majority_opinion = "2" if row['valence_1'] == minority_opinion else "1"

    
    score_unsafe_safe = row[f"parsed_score{idx_minority_opinion}to{idx_majority_opinion}"] 
    score_safe_unsafe = row[f"parsed_score{idx_majority_opinion}to{idx_minority_opinion}"]

    all_scores_unsafe_safe.append(score_unsafe_safe)
    all_scores_safe_unsafe.append(score_safe_unsafe)

In [149]:
# Count
print("Reverse scores in df_other:", count_reverse_scores(df_other))
print("Reverse scores in df_interest:", count_reverse_scores(df_interest))

Reverse scores in df_other: (24, 12.97)
Reverse scores in df_interest: (43, 22.87)


In [150]:
safeunsafe_higher_unsafesafe = len([i for i in range(len(all_scores_safe_unsafe)) if all_scores_safe_unsafe[i] > all_scores_unsafe_safe[i]])
unsafesafe_higher_safeunsafe = len([i for i in range(len(all_scores_safe_unsafe)) if all_scores_unsafe_safe[i] > all_scores_safe_unsafe[i]])
tot_len = len(all_scores_safe_unsafe)

print(f"Safe -> Unsafe higher than Unsafe -> Safe: {safeunsafe_higher_unsafesafe}/{tot_len} ({round(safeunsafe_higher_unsafesafe * 100 / tot_len, 2)}%)")
print(f"Scores are the same: {tot_len - safeunsafe_higher_unsafesafe - unsafesafe_higher_safeunsafe}/{tot_len} ({round((tot_len - safeunsafe_higher_unsafesafe - unsafesafe_higher_safeunsafe) * 100 / tot_len, 2)}%)")
print(f"Unsafe -> Safe higher than Safe -> Unsafe: {unsafesafe_higher_safeunsafe}/{tot_len} ({round(unsafesafe_higher_safeunsafe * 100 / tot_len, 2)}%)")

Safe -> Unsafe higher than Unsafe -> Safe: 29/188 (15.43%)
Scores are the same: 145/188 (77.13%)
Unsafe -> Safe higher than Safe -> Unsafe: 14/188 (7.45%)


- - - 
## Same analysis but including either as closer to dangerous or safe option.

In [168]:
# Other options (both either, both the same valence)
df_other = df_scores[(df_scores["valence_1"] == df_scores["valence_2"])]
# All other rows, row index use
df_interest = df_scores[df_scores.index.isin(df_other.index) == False]

In [170]:
# Enrich scores df with model alignment dat
# For each situation, add majoirity opiniony and majority opinion probability
majority_opinions = []
minority_opinions = []
majority_opinions_probs = []
minority_opinions_probs = []
for idx, row in df_interest.iterrows():
    situation = row["situation"]
    valence_1 = row["valence_1"]
    valence_2 = row["valence_2"]
    # If there is no either easy
    if valence_1 != "Either" and valence_2 != "Either":
        majority_opinion = model_alignment_data_reformatted[situation]["majority_opinion"]
        majority_opinion_prob = model_alignment_data_reformatted[situation]["majority_opinion_probability"]
        minority_opinion = model_alignment_data_reformatted[situation]["minority_opinion"]
        minority_opinion_prob = model_alignment_data_reformatted[situation]["minority_opinion_probability"]
    else:
        if majority_opinion in set([valence_1, valence_2]):
            # Either becomes the minority and majority remains the one with majority
            majority_opinion = majority_opinion
            majority_opinion_prob = model_alignment_data_reformatted[situation]["majority_opinion_probability"]
            minority_opinion = "Either"
            minority_opinion_prob = model_alignment_data_reformatted[situation]["minority_opinion_probability"]
        elif minority_opinion in set([valence_1, valence_2]):
            majority_opinion = "Either"
            majority_opinion_prob = model_alignment_data_reformatted[situation]["majority_opinion_probability"]
            minority_opinion = minority_opinion
            minority_opinion_prob = model_alignment_data_reformatted[situation]["minority_opinion_probability"]
        else:
            raise ValueError(f"Neither majority nor minority opinion found in valences for situation {situation}: valence_1={valence_1}, valence_2={valence_2}")

In [171]:
all_scores_safe_unsafe = []
all_scores_unsafe_safe = []
for idx, row in df_interest.iterrows():
    minority_opinion = row["minority_opinion"] # Proxy for danger
    idx_minority_opinion = "1" if row['valence_1'] == minority_opinion else "2"
    idx_majority_opinion = "2" if row['valence_1'] == minority_opinion else "1"

    
    score_unsafe_safe = row[f"parsed_score{idx_minority_opinion}to{idx_majority_opinion}"] 
    score_safe_unsafe = row[f"parsed_score{idx_majority_opinion}to{idx_minority_opinion}"]

    all_scores_unsafe_safe.append(score_unsafe_safe)
    all_scores_safe_unsafe.append(score_safe_unsafe)


In [172]:
# Count
print("Reverse scores in df_other:", count_reverse_scores(df_other))
print("Reverse scores in df_interest:", count_reverse_scores(df_interest))

Reverse scores in df_other: (4, 9.09)
Reverse scores in df_interest: (17, 17.89)


In [173]:
safeunsafe_higher_unsafesafe = len([i for i in range(len(all_scores_safe_unsafe)) if all_scores_safe_unsafe[i] > all_scores_unsafe_safe[i]])
unsafesafe_higher_safeunsafe = len([i for i in range(len(all_scores_safe_unsafe)) if all_scores_unsafe_safe[i] > all_scores_safe_unsafe[i]])
tot_len = len(all_scores_safe_unsafe)

print(f"Safe -> Unsafe higher than Unsafe -> Safe: {safeunsafe_higher_unsafesafe}/{tot_len} ({round(safeunsafe_higher_unsafesafe * 100 / tot_len, 2)}%)")
print(f"Scores are the same: {tot_len - safeunsafe_higher_unsafesafe - unsafesafe_higher_safeunsafe}/{tot_len} ({round((tot_len - safeunsafe_higher_unsafesafe - unsafesafe_higher_safeunsafe) * 100 / tot_len, 2)}%)")
print(f"Unsafe -> Safe higher than Safe -> Unsafe: {unsafesafe_higher_safeunsafe}/{tot_len} ({round(unsafesafe_higher_safeunsafe * 100 / tot_len, 2)}%)")

Safe -> Unsafe higher than Unsafe -> Safe: 13/95 (13.68%)
Scores are the same: 78/95 (82.11%)
Unsafe -> Safe higher than Safe -> Unsafe: 4/95 (4.21%)
